## 1. ADF Test
Standard stationarity tests (ADF, KPSS) lean heavily on autocorrelation structure, so data with a step-function shape (FFR, OPR, CPO) will most definitely show high persistence. 1. first differencing

In [2]:
import pandas as pd
import sys

sys.path.append('..')
from modules.source import Sources, parquet_daily, parquet_monthly


In [ ]:
"""
Variance question: how much of KLCI's monthly variance is explained by
Fed variables versus a domestic-only baseline?

Method: hierarchical (sequential) OLS regression.
1. Fit KLCI ~ domestic block                  -> R2_domestic
2. Fit KLCI ~ domestic block + Fed block       -> R2_full
Fed's share of variance = R2_full - R2_domestic

Standard errors are Newey-West HAC to
match the project's stated methodology; the Fed block's significance is
tested with a robust Wald F-test (a classical nested F-test assumes
i.i.d. errors, which HAC is explicitly correcting for).
"""

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tsa.stattools import adfuller

DOMESTIC = ["CPO", "Oil"]
FED = ["FFR_midpoint", Sources.rate_differential, "UST_10Y", "VIX", "DXY"]  # rate/carry, risk-appetite, FX/financial-conditions

df = pd.read_parquet(parquet_monthly)
change = df.xs("change", axis=1, level=1)

d = (
   change[["KLCI", "OPR", "Palm_Oil", "Brent_Oil"] + FED]
   .dropna()
   .rename(columns={"Palm_Oil": "CPO", "Brent_Oil": "Oil"})
)
n = len(d)
print(f"n = {n} months  ({d.index.min().date()} to {d.index.max().date()})")

# Newey-West plug-in lag: floor(4*(T/100)^(2/9))
L = int(np.floor(4 * (n / 100) ** (2 / 9)))
print(f"HAC (Newey-West) lag = {L}\n")


def fit_hac(rhs_vars):
   formula = f"KLCI ~ {' + '.join(rhs_vars)}"
   return smf.ols(formula, data=d).fit(cov_type="HAC", cov_kwds={"maxlags": L})


# ---------------------------------------------------------- 3 models
m_dom = fit_hac(DOMESTIC)
m_fed = fit_hac(FED)
m_full = fit_hac(DOMESTIC + FED)

r2 = pd.DataFrame(
   {
      "R2": [m_dom.rsquared, m_fed.rsquared, m_full.rsquared],
      "Adj R2": [m_dom.rsquared_adj, m_fed.rsquared_adj, m_full.rsquared_adj],
   },
   index=["Domestic only", "Fed only", "Domestic + Fed"],
)
print("R-SQUARED SUMMARY\n", r2, "\n")

incr_fed = m_full.rsquared - m_dom.rsquared   # Fed's incremental R2 (forward order)
incr_dom = m_full.rsquared - m_fed.rsquared   # Domestic's incremental R2 (reverse order)
overlap = m_dom.rsquared + m_fed.rsquared - m_full.rsquared

print(f"Fed's share of variance (domestic -> +Fed):   {incr_fed:.4f}")
print(f"Domestic's unique share (Fed -> +domestic):   {incr_dom:.4f}")
print(f"Shared / overlapping variance:                {overlap:.4f}\n")

# ------------------------------------------ robust joint-significance test
hypothesis = ", ".join(f"{v} = 0" for v in FED)
wald = m_full.wald_test(hypothesis, use_f=True, scalar=True)
print(
   "Wald test, Fed coefficients jointly zero (HAC-robust): "
   f"F({wald.df_num:.0f}, {wald.df_denom:.0f}) = {wald.statistic:.3f}, "
   f"p = {wald.pvalue:.3g}\n"
)

# ----------------------------------------------------------- coefficients
coefs = pd.DataFrame(
   {"coef": m_full.params, "HAC SE": m_full.bse, "p": m_full.pvalues}
).drop("Intercept")
std_y = d["KLCI"].std()
coefs["std_beta"] = [coefs.loc[v, "coef"] * d[v].std() / std_y for v in coefs.index]
coefs["sig"] = coefs["p"].apply(
   lambda p: "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else ""
)
print("FULL-MODEL COEFFICIENTS (HAC SE, standardized beta)\n", coefs.round(4), "\n")

# ------------------------------------------------------------------- VIF
X = m_full.model.exog
vif = pd.Series(
   [variance_inflation_factor(X, i) for i in range(X.shape[1])],
   index=m_full.model.exog_names,
).drop("Intercept")
print("VIF (full model)\n", vif.round(2), "\n")

# ------------------------------------------------------- stationarity check
print("ADF stationarity check")
for col in DOMESTIC + FED + ["KLCI"]:
   stat, p = adfuller(d[col], autolag="AIC", result_object=False)[:2]
   print(f"  {col:14s} ADF={stat:7.3f}  p={p:.3g}")


n = 139 months  (2015-02-28 to 2026-08-31)
HAC (Newey-West) lag = 4

R-SQUARED SUMMARY
                       R2    Adj R2
Domestic only   0.102603  0.082661
Fed only        0.294703  0.273649
Domestic + Fed  0.321179  0.284907 

Fed's share of variance (domestic -> +Fed):   0.2186
Domestic's unique share (Fed -> +domestic):   0.0265
Shared / overlapping variance:                0.0761

Wald test, Fed coefficients jointly zero (HAC-robust): F(4, 131) = 12.761, p = 8.45e-09

FULL-MODEL COEFFICIENTS (HAC SE, standardized beta)
                 coef  HAC SE       p  std_beta  sig
OPR          -0.0140  0.0196  0.4743   -0.0396     
CPO          -0.0167  0.0303  0.5800   -0.0418     
Oil           0.0416  0.0184  0.0237    0.1763   **
FFR_midpoint -0.0072  0.0133  0.5878   -0.0523     
UST_10Y       0.0128  0.0107  0.2332    0.0957     
VIX          -0.0016  0.0004  0.0001   -0.3053  ***
DXY          -0.5498  0.1323  0.0000   -0.3473  *** 

VIF (full model)
 OPR             1.30
CPO        